In [1]:
import surprise

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\filmtrust\ratings.txt",
                 sep=' ',names=['UserID','ItemID','Rating'])
df.sample(5)

,UserID,ItemID,Rating
23610,986,5,3.5
3029,147,246,2.5
9244,375,245,1.0
21196,877,17,3.0
17426,719,207,4.0


In [3]:
lowest_rating = df['Rating'].min()
higest_rating = df['Rating'].max()
lowest_rating,higest_rating

(np.float64(0.5), np.float64(4.0))

In [4]:
reader = surprise.Reader(rating_scale=(lowest_rating,higest_rating))
data = surprise.Dataset.load_from_df(df,reader)
type(data)

surprise.dataset.DatasetAutoFolds

## 'user_based': False -> item_based collaborative System

In [5]:
similarity_options = {'name':'cosine','user_based':True} 
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k=40
output = algo.fit(data.build_full_trainset())
# Calculates the  expected ratings for all the users 

Computing the cosine similarity matrix...
Done computing similarity matrix.


### Train Test Split

In [6]:
from surprise.model_selection import train_test_split

In [7]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=26)
algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [8]:
pred = algo.test(testset)
surprise.accuracy.rmse(pred)

RMSE: 0.9282


np.float64(0.9282056050129103)

### Tuning for best k

In [9]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [10]:
param_grid = { 'k':[25,40,50,65,75,85],
             'user_based':[False,True]
             }
kfold = KFold(n_splits=5, random_state=26,shuffle=True)
# gs = GridSearchCV(surprise.KNNBasic, param_grid,measures=['rmse','mae'], cv=kfold)
gs = GridSearchCV(surprise.KNNWithZScore, param_grid,measures=['rmse','mae'], cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [76]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.9761915086082429
{'k': 25, 'user_based': False}
0.7698381577357556
{'k': 25, 'user_based': False}


### Building best model on whole data

In [12]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [13]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)  # -> elements of a which not present in b

array([2, 5, 8])

In [14]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(b,a)  # -> elements of b which not present in a

array([], dtype=int64)

In [15]:
# Total Numbers of item
ItemID = df['ItemID'].unique()
len(ItemID)

2071

In [16]:
user = 50
u_iid = df[df['UserID'] == user ]['ItemID'].unique()
print('List of item rated by user :\n',u_iid)
print('No. of items rated by user {0} : {1}'.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(ItemID,u_iid)
print('Items not rated by the user or those items which the expected ratings are to be predicted :\n',iids_to_predict)

List of item rated by user :
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39
Items not rated by the user or those items which the expected ratings are to be predicted :
 [  14   15   16 ... 2069 2070 2071]


In [54]:
testset = [[user,iid,0.] for iid in iids_to_predict]

In [18]:
prediction = algo.test(testset)

In [19]:
exp_rating = [ (prediction[i].iid,prediction[i].est) for i in range(0,len(prediction)) ]
df_rating = pd.DataFrame(exp_rating , columns=['ItemID','Est_Rating'])
df_rating.sort_values('Est_Rating',ascending=False).head(30)

,ItemID,Est_Rating
12,27,4.0
2014,2054,4.0
649,689,4.0
697,737,4.0
53,68,4.0
732,772,4.0
734,774,4.0
506,546,4.0
786,826,4.0
804,844,4.0


# Exercise

In [20]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\ml-100k\u.data",
                 sep='\t',names=['UserId','ItemID','Rating','TS'])
df.sample(5)

,UserId,ItemID,Rating,TS
52229,457,9,5,882393485
94591,627,526,4,879529916
49974,17,126,4,885272724
85630,56,393,4,892677047
56441,435,1,5,884131712


In [21]:
df.drop('TS',axis=1,inplace=True)

In [22]:
df.sample(2)

,UserId,ItemID,Rating
93837,535,156,2
60555,697,979,5


In [23]:
min_rate = df['Rating'].min()
max_rate = df['Rating'].max()

In [24]:
reader = surprise.Reader(rating_scale=(min_rate,max_rate))
data = surprise.Dataset.load_from_df(df,reader)

In [25]:
param_gird = {
    'k':[25,40,50,60,65,70,85],
    'user_based':[True,False]
}
kfold = KFold(n_splits=5,shuffle=True,random_state=26)
gs = GridSearchCV(surprise.KNNBasic,param_grid,cv=kfold,measures=['rmse','mae'])
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [26]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.9761915086082429
{'k': 25, 'user_based': False}
0.7698381577357556
{'k': 25, 'user_based': False}


In [27]:
model = gs.best_estimator['rmse']
model.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [28]:
ItemID = df['ItemID'].unique()
len(ItemID)

1682

In [58]:
user = 50
u_iid = df[df['UserId'] == user ]['ItemID'].unique()
print('List of item rated by user :\n',u_iid)
print('No. of items rated by user {0} : {1}'.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(ItemID,u_iid)
print('Items not rated by the user or those items which the expected ratings are to be predicted :\n',iids_to_predict)

List of item rated by user :
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50 : 24
Items not rated by the user or those items which the expected ratings are to be predicted :
 [   1    2    3 ... 1680 1681 1682]


In [71]:
testset = [[user,iid,0.] for iid in iids_to_predict]
prediction = algo.test(testset)

In [72]:
exp_rating = [ (prediction[i].iid,prediction[i].est) for i in range(0,len(prediction)) ]
df_rating = pd.DataFrame(exp_rating , columns=['ItemID','Est_Rating'])
df_rating.sort_values('Est_Rating',ascending=False).head(30)

,ItemID,Est_Rating
1652,1677,4.0
930,952,4.0
332,349,4.0
900,922,4.0
1559,1584,4.0
65,68,4.0
991,1015,4.0
872,894,4.0
869,891,4.0
866,888,4.0


In [73]:
movies = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\ml-100k\movies_list.csv",
                    encoding='latin-1')
movies.head()

,movie id,movie title,release date,video release date,IMDb URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [74]:
movies.columns = movies.columns.str.strip()
movies.columns 

Index(['movie id', 'movie title', 'release date', 'video release date',
       'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [75]:
rating = df_rating.merge(movies[['movie id','movie title','release date']],
                              how ='left',left_on='ItemID',right_on='movie id')
rating.sort_values('Est_Rating',ascending=False).head(10)

,ItemID,Est_Rating,movie id,movie title,release date
1652,1677,4.0,1677,Sweet Nothing (1995),20-Sep-96
930,952,4.0,952,Blue in the Face (1995),01-Jan-95
332,349,4.0,349,Hard Rain (1998),16-Jan-98
900,922,4.0,922,Dead Man (1995),10-May-96
1559,1584,4.0,1584,"Symphonie pastorale, La (1946)",01-Jan-46
65,68,4.0,68,"Crow, The (1994)",01-Jan-94
991,1015,4.0,1015,Shiloh (1997),23-May-97
872,894,4.0,894,Home Alone 3 (1997),01-Jan-97
869,891,4.0,891,Bent (1997),18-Dec-97
866,888,4.0,888,One Night Stand (1997),01-Jan-97
